# Getting started with Claude Opus 4.7 on Amazon Bedrock

**Anthropic's most capable Opus model yet — advancing agentic coding, knowledge work, and long-running tasks**

This notebook demonstrates how to use Claude Opus 4.7 on Amazon Bedrock. Opus 4.7 is a direct upgrade from Opus 4.6, with improvements across agentic coding, professional knowledge work, long-horizon tasks, cross-session memory, and high-resolution vision.

**New with Opus 4.7:** 
- **Enhanced performance:** 64.3% on SWE-bench Pro, 69.4% on Terminal-Bench 2.0, 63.4% on Finance Agent
- **Improved safety:** Enhanced safeguards including the new Cyber Verification Program
- **Higher-resolution vision:** Support for 2576px images (up from 1568px) for finer detail in screenshots, diagrams, and design files
- **No pricing jump:** Full 1M context with no pricing increase above 200K tokens
- **Data retention:** Amazon Bedrock may store inputs and outputs for up to 30 days for automated abuse detection, securely within the ZOA environment in your selected AWS Region

---

## What you'll learn

- **Basic usage**: Get started with Claude Opus 4.7 using the Converse, InvokeModel, and Messages APIs
- **Adaptive thinking**: Leverage Claude's intelligent extended thinking mode with configurable effort levels
- **Task budgets**: Set token spending limits for agentic loops (beta)
- **Context compaction**: Enable infinite context for long-running conversations (beta)

---

## When to use Claude Opus 4.7

Claude Opus 4.7 is ideal for:

- **Agentic coding** — stronger performance on long-horizon autonomy, systems engineering, and complex code reasoning
- **Knowledge work** — document creation, financial analysis, and multi-step research workflows
- **Long-running tasks** — multi-step tool use, error recovery, and reasoning across the full 1M-token context
- **Autonomous agents** — better reasoning through ambiguity and stronger memory for longer unsupervised runs
- **Vision tasks** — high-resolution image support for screenshots, diagrams, and design files
- **Cybersecurity** — enhanced safeguards for security workflows with the Cyber Verification Program

---

## Key capabilities

| Capability | Details |
|------------|--------|
| **Context window** | Up to 1M tokens |
| **Max output** | Up to 128K tokens |
| **Knowledge cutoff** | January 2026 |
| **Adaptive thinking** | Extended reasoning that activates when needed, with configurable effort levels |
| **Task budgets** | Set token spending limits for agentic loops (beta) |
| **Context compaction** | Automatic summarization for infinite context (beta, InvokeModel and Messages API) |
| **Memory** | Cross-session memory — knows when to write context down and when to recall it |
| **Vision** | High-resolution image input (2576px) for finer detail in screenshots, diagrams, and design files |

---

## API options on Amazon Bedrock

Claude Opus 4.7 is available through **Amazon Bedrock Mantle**, a new endpoint that exposes multiple API paths. You can invoke the model through multiple API paths:

| API | Endpoint | When to use |
|-----|----------|-------------|
| **Messages API** | `bedrock-mantle.{region}.api.aws/anthropic/v1/messages` | Direct access to Anthropic's native API format via Bedrock Mantle. Full feature support. |
| **Responses API** | `bedrock-mantle.{region}.api.aws/anthropic/v1/responses` | **Coming soon.** Streamlined API for most use cases. |
| **Chat Completions API** | `bedrock-mantle.{region}.api.aws/v1/chat/completions` | **Coming soon.** OpenAI-compatible API format. |
| **InvokeModel API** | `bedrock-runtime.{region}.amazonaws.com` | Bedrock's native API with Anthropic request body format. Supports all features including compaction. |
| **Converse API** | `bedrock-runtime.{region}.amazonaws.com` | Bedrock's unified conversational API. Simplest integration, works across Bedrock models. |

The Messages API is the recommended starting point for new customers and provides complete feature parity with Anthropic's first-party API. The InvokeModel and Converse APIs go through the standard `bedrock-runtime` endpoint and are mapped internally by Bedrock Mantle.

---

## 1. Setup and configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install boto3 --upgrade
# !pip install anthropic[bedrock] --upgrade  # For Messages and Responses APIs via Bedrock Mantle


In [ ]:
import boto3
import json
import base64
from botocore.config import Config

# Try importing Anthropic Bedrock Mantle client (for Messages/Responses APIs)
try:
    from anthropic import AnthropicBedrockMantle

    MANTLE_AVAILABLE = True
except ImportError:
    print(
        "Note: anthropic[bedrock] not installed. "
        "Messages and Responses API examples will be skipped."
    )
    MANTLE_AVAILABLE = False

In [ ]:
import boto3
import json
from botocore.config import Config

# Runtime configuration (for Converse and InvokeModel APIs)
RUNTIME_REGION = "us-west-2"
RUNTIME_MODEL_ID = "us.anthropic.claude-opus-4-7"

# Mantle configuration (for Messages and Responses APIs)
MANTLE_REGION = "us-west-2"
MANTLE_MODEL_ID = "anthropic.claude-opus-4-7"
MANTLE_BASE_URL = f"https://bedrock-mantle.{MANTLE_REGION}.api.aws"

# Config with longer timeout for thinking/compaction
FAST_CONFIG = Config(read_timeout=1200, retries={"max_attempts": 1})

# Initialize session
session = boto3.Session()

# Initialize the Bedrock Runtime client
bedrock_runtime = session.client(
    service_name="bedrock-runtime",
    region_name=RUNTIME_REGION,
    config=FAST_CONFIG,
)

print(f"✓ Runtime Region: {RUNTIME_REGION}")
print(f"✓ Runtime Model: {RUNTIME_MODEL_ID}")
print(f"✓ Mantle Region: {MANTLE_REGION}")
print(f"✓ Mantle Model: {MANTLE_MODEL_ID}")
print(f"✓ Mantle Base URL: {MANTLE_BASE_URL}")
print("✓ Client initialized successfully")

---

## 2. Basic usage with Converse API

The Converse API provides a unified interface for conversational AI across Amazon Bedrock models. This is the simplest way to get started.

In [ ]:
# Basic request using the Converse API
try:
    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "text": (
                            "What are three key considerations when designing "
                            "a distributed system for high availability?"
                        )
                    }
                ],
            }
        ],
        inferenceConfig={"maxTokens": 2048},
    )

    response_text = response["output"]["message"]["content"][0]["text"]
    print(response_text[:800] + "..." if len(response_text) > 800 else response_text)
    print(
        f"\nUsage: Input tokens={response['usage']['inputTokens']}, "
        f"Output tokens={response['usage']['outputTokens']}"
    )

except Exception as e:
    print(f"Error with Converse API: {e}")

---

## 3. Basic usage with InvokeModel API

The InvokeModel API provides direct access to Claude's native request format and supports the full set of Opus 4.7 features.

In [ ]:
# Basic request using the InvokeModel API
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 2048,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Explain the CAP theorem and its implications for database design."
                    ),
                }
            ],
        }
    ],
}

try:
    response = bedrock_runtime.invoke_model(
        modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body)
    )

    response_body = json.loads(response["body"].read())
    response_text = response_body["content"][0]["text"]
    print(response_text[:800] + "..." if len(response_text) > 800 else response_text)
    print(
        f"\nUsage: Input tokens={response_body['usage']['input_tokens']}, "
        f"Output tokens={response_body['usage']['output_tokens']}"
    )

except Exception as e:
    print(f"Error with InvokeModel API: {e}")

---

## 4. Basic usage with Messages API (Bedrock Mantle)

The Messages API is available through the Bedrock Mantle endpoint. It uses the same request/response format as Anthropic's first-party API, authenticated with AWS SigV4. This is the most direct way to access Claude's full capabilities.

You can use the `anthropic[bedrock]` SDK package for a streamlined experience:

In [ ]:
# Messages API example using Bedrock Mantle
if MANTLE_AVAILABLE:
    # Initialize the Bedrock Mantle client (uses SigV4 auth automatically)
    mantle_client = AnthropicBedrockMantle(
        aws_region=MANTLE_REGION,
        base_url=f"{MANTLE_BASE_URL}/anthropic",
    )

    try:
        # Create a message using the Messages API
        response = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "What are the key differences between eventual and strong "
                        "consistency in distributed systems?"
                    ),
                }
            ],
        )

        print(response.content[0].text)
        print(
            f"\nUsage: Input tokens={response.usage.input_tokens}, "
            f"Output tokens={response.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error with Messages API: {e}")
else:
    print("Skipping Messages API example - anthropic[bedrock] not installed")

---

## 5. Responses API (Coming Soon)

The Responses API will be the **recommended default for new customers** when it becomes available. It will provide a streamlined interface optimized for most use cases while maintaining full access to Claude's capabilities.

**Status**: Coming soon to Amazon Bedrock Mantle.

---

## 5. Adaptive thinking

Adaptive thinking gives Claude the freedom to think if and when it determines reasoning is required. Instead of fixed token budgets, you guide Claude's reasoning depth with an `effort` parameter.

### Effort levels

| Effort | Description |
|--------|-------------|
| **max** | Maximum reasoning depth. Reserve for the most complex analytical workloads. |
| **high** | Default. Deep reasoning on complex tasks. Start here. |
| **medium** | Balanced. May skip thinking for straightforward queries. |
| **low** | Minimal thinking, prioritizes speed and cost. |

### Adaptive thinking with Messages API

With the Messages API, adaptive thinking is configured using the thinking dict, the effort parameter goes in output_config (via extra_body)

In [ ]:
# Adaptive thinking with Messages API (Bedrock Mantle)
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,  # Using MANTLE_MODEL_ID variable
            max_tokens=50000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": "high"}},
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Design a fault-tolerant distributed consensus algorithm "
                        "that handles Byzantine failures in an asynchronous "
                        "network with up to 1/3 faulty nodes."
                    ),
                }
            ],
        )

        # Check if thinking was triggered
        has_thinking = any(block.type == "thinking" for block in message.content)
        print(f"Claude decided to think: {has_thinking}\n")

        # Display content blocks
        for block in message.content:
            if block.type == "thinking":
                print("Thinking (first 300 chars):")
                print(block.thinking[:300] + "...\n")
            elif block.type == "text":
                print("Response:")
                text = block.text
                print(text[:800] + "..." if len(text) > 800 else text)

        print(
            f"\nUsage: Input tokens={message.usage.input_tokens}, "
            f"Output tokens={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error with Messages API adaptive thinking: {e}")
else:
    print("Skipping Messages API example - anthropic[bedrock] not installed")

### Comparing effort levels

Different effort levels produce different reasoning depths. Higher effort may trigger more thinking on complex prompts:

In [ ]:
# Compare effort levels on the same prompt (Messages API)
def test_effort_level(effort_level: str, prompt: str):
    if not MANTLE_AVAILABLE:
        return {
            "effort": effort_level,
            "thought": False,
            "thinking_chars": 0,
            "response_chars": 0,
            "success": False,
            "error": "anthropic[bedrock] not installed",
        }

    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,  # Using MANTLE_MODEL_ID variable
            max_tokens=12000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": effort_level}},
            messages=[{"role": "user", "content": prompt}],
        )

        has_thinking = False
        thinking_length = 0
        response_length = 0

        for block in message.content:
            if block.type == "thinking":
                has_thinking = True
                thinking_length = len(block.thinking)
            elif block.type == "text":
                response_length = len(block.text)

        return {
            "effort": effort_level,
            "thought": has_thinking,
            "thinking_chars": thinking_length,
            "response_chars": response_length,
            "success": True,
        }
    except Exception as e:
        return {
            "effort": effort_level,
            "thought": False,
            "thinking_chars": 0,
            "response_chars": 0,
            "success": False,
            "error": str(e),
        }


prompt = (
    "What are the key considerations when implementing a caching strategy "
    "for a high-traffic API?"
)
print("Testing different effort levels on the same prompt...\n")

for effort in ["low", "medium", "high", "max"]:
    result = test_effort_level(effort, prompt)
    if result["success"]:
        print(
            f"Effort: {result['effort']:6} | Thought: {str(result['thought']):5} | "
            f"Thinking: {result['thinking_chars']:5} chars | "
            f"Response: {result['response_chars']:5} chars"
        )
    else:
        print(f"Effort: {result['effort']:6} | Error: {result['error']}")

print("\nHigher effort levels may trigger more thinking on complex prompts")

---

## 6. Task Budgets (beta)

Task Budgets are a new Opus 4.7 feature that allow you to set token spending limits for entire agentic loops. This prevents runaway costs during autonomous operations while still allowing Claude to complete complex multi-step tasks.

**How it works:**
- Set a maximum token budget via `output_config.task_budget`
- Claude tracks token usage across the entire conversation or agentic loop
- When the budget is reached, Claude will wrap up gracefully rather than stopping mid-task
- Ideal for autonomous agents, long-running workflows, and cost-controlled environments

In [ ]:
# Task Budget example - limit token spending for agentic loops (Messages API)
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,  # Using MANTLE_MODEL_ID variable
            max_tokens=4096,
            thinking={"type": "adaptive"},
            extra_body={
                "anthropic_beta": ["task-budgets-2026-03-13"],
                "output_config": {"task_budget": {"type": "tokens", "total": 25000}},
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Plan and execute a comprehensive market research project "
                        "for a new fintech startup targeting small businesses. "
                        "Include competitor analysis, market sizing, and "
                        "go-to-market strategy recommendations."
                    ),
                }
            ],
        )

        # Check task budget usage
        task_budget_used = getattr(message.usage, "task_budget_tokens_used", 0)
        print(f"Task budget tokens used: {task_budget_used}")

        for block in message.content:
            if block.type == "text":
                response_text = block.text
                print(
                    f"\nResponse:\n{response_text[:800]}..."
                    if len(response_text) > 800
                    else f"\nResponse:\n{response_text}"
                )

        print(
            f"\nUsage: Input tokens={message.usage.input_tokens}, "
            f"Output tokens={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error with Task Budget example: {e}")
        print(
            "Note: Task Budgets require the beta header and may not be "
            "available in all regions yet."
        )
else:
    print("Skipping Task Budget example - anthropic[bedrock] not installed")

---

## 7. Context compaction (beta)

Context compaction enables "infinite context" for long-running conversations and agentic tasks by automatically summarizing older context when approaching the context window limit.

**How it works:**
1. Enable compaction with `context_management.edits` containing a `compact_20260112` edit
2. When input tokens exceed a trigger threshold, Claude generates a summary
3. The API returns a compaction content block
4. Pass the compaction block back as part of the messages array on subsequent requests

**Important:**
- Context compaction is available with both the **InvokeModel and Messages APIs** (not Converse API)
- Requires `anthropic_beta: ["compact-2026-01-12"]` header
- Default trigger is ~500K tokens; customize with `trigger` parameter

In [ ]:
# Basic compaction example
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=256,
            extra_body={
                "anthropic_beta": ["compact-2026-01-12"],
                "context_management": {
                    "edits": [
                        {
                            "type": "compact_20260112",
                            "trigger": {
                                "type": "input_tokens",
                                "value": 100000,  # Won't trigger on this short message
                            },
                        }
                    ]
                },
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Let's start a long-running conversation about building a "
                        "complex e-commerce platform."
                    ),
                }
            ],
        )

        # Check for compaction block
        has_compaction = any(
            getattr(b, "type", None) == "compaction" for b in message.content
        )
        print(f"Compaction triggered: {has_compaction}")

        for block in message.content:
            if block.type == "text":
                response_text = block.text
                print(
                    f"\nResponse:\n{response_text[:800]}..."
                    if len(response_text) > 800
                    else f"\nResponse:\n{response_text}"
                )
            elif block.type == "compaction":
                print(
                    "\nCompaction block received — pass this back on subsequent requests"
                )

        print(
            f"\nUsage: Input tokens={message.usage.input_tokens}, "
            f"Output tokens={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error with Compaction example: {e}")
        print(
            "Note: Context compaction requires the beta header and may not be "
            "available in all regions yet."
        )

In [ ]:
# Multi-turn conversation with compaction and custom trigger (Messages API)
if MANTLE_AVAILABLE:
    messages = []

    def chat(user_message: str):
        """Send a message and handle compaction automatically"""
        messages.append({"role": "user", "content": user_message})

        try:
            message = mantle_client.messages.create(
                model=MANTLE_MODEL_ID,  # Using MANTLE_MODEL_ID variable
                max_tokens=4096,
                messages=messages,
                extra_body={
                    "anthropic_beta": ["compact-2026-01-12"],
                    "context_management": {
                        "edits": [
                            {
                                "type": "compact_20260112",
                                "trigger": {
                                    "type": "input_tokens",
                                    "value": 100000,  # Trigger at 100K tokens
                                },
                            }
                        ]
                    },
                },
            )

            # Add assistant response to conversation history
            # Convert message.content to the format expected for messages array
            assistant_content = []
            for block in message.content:
                if block.type == "text":
                    assistant_content.append({"type": "text", "text": block.text})
                elif block.type == "compaction":
                    assistant_content.append(
                        {"type": "compaction", "compaction": block.compaction}
                    )

            messages.append({"role": "assistant", "content": assistant_content})

            # Check for compaction
            has_compaction = any(
                getattr(b, "type", None) == "compaction" for b in message.content
            )
            if has_compaction:
                print("Compaction occurred in this turn")

            # Return text response
            for block in message.content:
                if block.type == "text":
                    return block.text
            return ""

        except Exception as e:
            print(f"Error in chat turn: {e}")
            return f"Error: {e}"

    # Have a multi-turn conversation
    try:
        print("Turn 1:")
        response1 = chat("Design the database schema for an e-commerce platform")
        print(response1[:500] + "..." if len(response1) > 500 else response1)
        print("\n" + "=" * 50 + "\n")

        print("Turn 2:")
        response2 = chat("Now add the authentication and authorization system")
        print(response2[:500] + "..." if len(response2) > 500 else response2)

        print(f"\nConversation has {len(messages)} message(s)")

    except Exception as e:
        print(f"Error in multi-turn conversation: {e}")
else:
    print("Skipping multi-turn compaction example - anthropic[bedrock] not installed")

---

## 8. High-resolution vision capabilities

Opus 4.7 supports higher-resolution image input (2576px up from 1568px), providing enhanced accuracy on charts, dense documents, screenshots, and complex diagrams. This capability is automatically available with no API changes needed.

In [ ]:
# High-resolution vision example (Messages API)
def encode_image(image_path):
    """Encode image to base64 string"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# Vision request using Messages API
if MANTLE_AVAILABLE:
    print(
        "Claude can now analyze high-resolution images up to 2576px "
        "with enhanced detail recognition"
    )
    print("Perfect for:")
    print("- Screenshot analysis")
    print("- Complex diagram interpretation")
    print("- Dense document processing")
    print("- Design file review")

    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4096,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/png",
                                "data": encode_image("path/to/your/image.png"),
                            },
                        },
                        {
                            "type": "text",
                            "text": (
                                "Analyze this screenshot and provide detailed "
                                "feedback on the UI/UX design. Focus on layout, "
                                "accessibility, and user flow improvements."
                            ),
                        },
                    ],
                }
            ],
        )
        
        response_text = message.content[0].text
        print(
            f"\nVision Analysis:\n{response_text[:800]}..."
            if len(response_text) > 800
            else f"\nVision Analysis:\n{response_text}"
        )
        print(
            f"\nUsage: Input tokens={message.usage.input_tokens}, "
            f"Output tokens={message.usage.output_tokens}"
        )



    except Exception as e:
        print(f"Error with Vision API: {e}")
else:
    print("Skipping Vision example - anthropic[bedrock] not installed")

---

## 9. Next steps

- Explore [Claude Opus 4.7 documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/) on Amazon Bedrock
- Try different effort levels on your specific workloads to find the right balance of quality vs. speed
- For long-running agents, enable compaction to avoid context window limits
- Check out the [Anthropic SDK for Bedrock Mantle](https://docs.anthropic.com/en/api/claude-on-amazon-bedrock) for Messages API access